# 2 — Spatial niches: components, boundaries, composition

Grouping cells into spatial niches, drawing boundaries around them, and
summarising what each contains. Runs on synthetic data.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

rng = np.random.default_rng(0)

def synthetic_tissue(n_per_niche=120, n_niches=3, seed=0):
    """Three spatially separated cell niches with two marker features."""
    rng = np.random.default_rng(seed)
    centres = np.array([[0.0, 0.0], [600.0, 0.0], [0.0, 600.0]])[:n_niches]
    coords, niche = [], []
    for i, c in enumerate(centres):
        coords.append(c + rng.normal(0, 60, (n_per_niche, 2)))
        niche += [f'niche_{i}'] * n_per_niche
    coords = np.vstack(coords)
    n = len(coords)
    obs = pd.DataFrame({
        'X_centroid': coords[:, 0],
        'Y_centroid': coords[:, 1],
        'imageid': 'demo_image',
        'label': np.arange(n),
        'niche': niche,
        'phenotype': rng.choice(['duct', 'immune', 'stroma'], n, p=[.4, .35, .25]),
        'area': rng.lognormal(4.6, 0.3, n),
        'nc_ratio': rng.uniform(0.15, 0.75, n),
        # a feature with real spatial structure, for the statistics below
        'gradient': coords[:, 0] + coords[:, 1] + rng.normal(0, 30, n),
        'noise': rng.normal(size=n),
    })
    adata = ad.AnnData(X=rng.lognormal(0, 1, (n, 2)))
    adata.var_names = ['CD8', 'Ki67']
    adata.obs = obs
    adata.obs_names = [f'cell_{i}' for i in range(n)]
    return adata

adata = synthetic_tissue()
adata

## Detect spatial components

Cells carrying the target label are grouped by spatial proximity. The
three synthetic niches are 600 units apart, so a 200-unit linking radius
recovers them exactly.

In [ ]:
import spatioev as sv

adata.obs['target'] = 'tissue'
out = sv.tl.cluster_spatial_components(
    adata, label_key='target', label_value='tissue', radius=200
)
out.obs['tissue_component'].value_counts()

## Boundaries

`build_niche_boundaries` returns one polygon per component, with its
area and bounding box. `concave_hull` follows irregular shapes; here we
use `convex_hull` for a simple, exactly checkable answer.

In [ ]:
boundaries = sv.tl.build_niche_boundaries(
    out,
    component_key='tissue_component',
    min_cluster_size=20,
    method='convex_hull',
)
boundaries[['tissue_component', 'n_cells', 'area']]

Buffering adds `expanded_geometry` / `shrunk_geometry` columns and leaves
the original geometry untouched, so core and rim can be compared.

In [ ]:
buffered = sv.tl.buffer_niche_boundaries(
    boundaries, component_key='tissue_component', expand_by=40.0, shrink_by=25.0
)
[round(g.area, 1) for g in buffered['expanded_geometry']]

## Assign cells to regions

Each cell is placed in the core, an inner/outer border, or dropped if it
falls outside every geometry.

In [ ]:
regions = sv.tl.assign_cells_to_niche_regions(
    out, buffered, component_key='tissue_component', mode='buffer'
)
regions['region'].value_counts()

## Composition

Phenotype make-up per niche and region.

In [ ]:
comp = sv.tl.summarize_niche_composition(
    out, regions, component_key='tissue_component', phenotype_key='phenotype'
)
comp.head(9)